In [16]:
import pandas as pd
import re
from datetime import datetime
import json

raw_df = pd.read_csv("csv_outputs/merged_data_2026-01-14.csv", dtype="string")


In [17]:
raw_df["company_name"].value_counts().to_csv("unique_company_names.csv", index=False)

In [18]:
# cleaning data - 30s
df = raw_df.copy()

strip_cols = ["company_address", "home_address"]
for col in strip_cols:
    df[col] = df[col].str.strip()

# make the column values lowercase
lowercase_cols = ["email", "linkedin", "company_linkedin", "facebook", "twitter"]
for col in lowercase_cols:
    df[col] = df[col].str.lower()

# make the gender column uppercase
uppercase_cols = ["gender"]
for col in uppercase_cols:
    df[col] = df[col].str.upper()

df["gender"] = df["gender"].replace({"MALE": "M", "FEMALE": "F"})

# columns to title case
title_cols = ["first_name", "last_name", "job_title", "industry"]
for col in title_cols:
    df[col] = df[col].str.title()

# remove all phone numbers that contain e+ or are less than 6z digits
for col in ["mobile_phone", "work_phone", "landline_phone"]:
    mask = df[col].astype(str).str.lower().str.contains('e+', na=False)
    mask = mask | (df[col].astype(str).str.len() <= 6)
    df.loc[mask, col] = None

# remove all email values where it does not contain @
df.loc[~df["email"].astype(str).str.contains("@", na=False), "email"] = None

url_pattern = r'(https?:/|www\.)'
cols_to_check_for_links = [x for x in df.columns.tolist() if x not in ['source_file', 'record_id', 'company_linkedin', 'facebook', 'twitter', 'company_website', 'linkedin']]
for col in cols_to_check_for_links:
    df.loc[df[col].str.contains(url_pattern, case=False, na=False, regex=True), col] = None

range_mask = df['company_employee_size_actual'].str.contains(r'^\d+\s*-\s*\d+$', na=False, regex=True)
df.loc[range_mask, 'company_employee_size_range'] = df.loc[range_mask, 'company_employee_size_actual']
df.loc[range_mask, 'company_employee_size_actual'] = None



C:\Users\rayle\AppData\Local\Temp\ipykernel_12248\1516021684.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df.loc[df[col].str.contains(url_pattern, case=False, na=False, regex=True), col] = None
C:\Users\rayle\AppData\Local\Temp\ipykernel_12248\1516021684.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df.loc[df[col].str.contains(url_pattern, case=False, na=False, regex=True), col] = None
C:\Users\rayle\AppData\Local\Temp\ipykernel_12248\1516021684.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df.loc[df[col].str.contains(url_pattern, case=False, na=False, regex=True), col] = None
C:\Users\rayle\AppData\Local\Temp\ipykernel_12248\1516021684.py:37: UserWarning: This pattern is interpreted as a regular expre

In [19]:
# cleaning data - 40s

# Extract 5-digit zipcode root and 4-digit zipfour from company_zipcode.
# us_zips = pd.read_csv("uszips.csv", dtype="string")["zip"].unique().tolist()
df["company_zipfour"] = df["company_zipfour"].fillna(df["company_zipcode"].astype(str).str.findall(r'(?<!\d)(\d{4})(?!\d)').str[-1]).fillna(df["company_address"].str.findall(r'(?<!\d)(\d{4})(?!\d)').str[-1])
df["company_zipfour"] = "'" + df["company_zipfour"].str.rjust(4, "0")

df["company_zipcode"] = df["company_zipcode"].astype(str).str.findall(r'(\d{5})').str[-1].combine_first(df["company_zipcode"]).fillna(df["company_address"].str.findall(r'(\d{5})').str[-1])
df["company_zipcode"] = "'" + df["company_zipcode"].str.rjust(5, "0")

# df.loc[(~df["company_zipcode"].isin(us_zips)) & (df["company_zipcode"].notnull()), "company_zipcode"] = None
df.loc[(df["company_address"].notna() & df["company_zipfour"].notna() & df.apply(lambda row: str(row["company_address"]).startswith(str(row["company_zipfour"])), axis=1)), "company_zipfour"] = None

df["home_zipfour"] = df["home_zipfour"].fillna(df["home_zipcode"].astype(str).str.findall(r'(?<!\d)(\d{4})(?!\d)').str[-1]).fillna(df["home_address"].str.findall(r'(?<!\d)(\d{4})(?!\d)').str[-1])
df["home_zipfour"] = "'" + df["home_zipfour"].str.rjust(4, "0")
df["home_zipcode"] = df["home_zipcode"].astype(str).str.findall(r'(\d{5})').str[-1].combine_first(df["home_zipcode"]).fillna(df["home_address"].str.findall(r'(\d{5})').str[-1])
df["home_zipcode"] = "'" + df["home_zipcode"].str.rjust(5, "0")
# df.loc[(~df["home_zipcode"].isin(us_zips)) & (df["home_zipcode"].notnull()), "home_zipcode"] = None
df.loc[(df["home_address"].notna() & df["home_zipfour"].notna() & df.apply(lambda row: str(row["home_address"]).startswith(str(row["home_zipfour"])), axis=1)), "home_zipfour"] = None


In [20]:
cities_states = pd.read_csv("uscities_usstates.csv")
counties = pd.read_csv("uscounties.csv")

state_to_name_mapping = cities_states.set_index("state_id")["state_name"].to_dict()
us_states = pd.concat([cities_states["state_id"], cities_states["state_name"]]).str.upper().unique().tolist()
us_counties = counties["county_ascii"].str.upper().unique().tolist()
us_cities = cities_states["city_ascii"].str.upper().unique().tolist()
us_cities = [city for city in us_cities if city not in us_states]


In [21]:
# cleaning data - 20s

# Function to extract US state (full name or abbreviation) from address
state_pattern = r"\b(" + "|".join(us_states) + r")\b"
df.loc[~df["company_state"].str.upper().str.match(state_pattern, na=False), "company_state"] = None
df.loc[~df["home_state"].str.upper().str.match(state_pattern, na=False), "home_state"] = None
# Fix company_state - move city names to city field
city_names_in_state = ['oklahoma city', 'la porte', 'idaho falls', 'kansas city', 'missouri city', 'la vista', 'ohio city', 'la marque', 'la crescenta-montrose', 'texas city', 'virginia beach', 'de pere', 'la salle', 'oregon city', 'colorado springs']
state_mask = df['company_state'].str.lower().isin(city_names_in_state)
df.loc[state_mask, 'company_city'] = df.loc[state_mask, 'company_state']
df.loc[state_mask, 'company_state'] = None
df["company_state"] = df["company_state"].fillna(df["company_address"].str.upper().str.findall(state_pattern).str[-1]).fillna(df["company_location"].str.upper().str.findall(state_pattern).str[-1])
df["home_state"] = df["home_state"].fillna(df["home_address"].str.upper().str.findall(state_pattern).str[-1])
df["company_state"] = df["company_state"].str.upper().replace(state_to_name_mapping).str.title()
df["home_state"] = df["home_state"].str.upper().replace(state_to_name_mapping).str.title()


In [22]:
# 18mins

city_pattern = r"\b(" + "|".join(us_cities) + r")\b"
df.loc[~df["company_city"].str.upper().str.match(city_pattern, na=False), "company_city"] = None
df.loc[~df["home_city"].str.upper().str.match(city_pattern, na=False), "home_city"] = None
df["company_city"] = df["company_city"].fillna(df["company_address"].str.upper().str.findall(city_pattern).str[-1]).fillna(df["company_location"].str.upper().str.findall(city_pattern).str[-1]).str.title()
df["home_city"] = df["home_city"].fillna(df["home_address"].str.upper().str.findall(city_pattern).str[-1])


In [23]:
county_pattern = r"\b(" + "|".join(us_counties) + r")\b"
df.loc[~df["county"].str.upper().str.match(county_pattern, na=False), "county"] = None
# df["county"] = df["county"].fillna(df["company_address"].str.upper().str.findall(county_pattern).str[-1])
# df["county"] = df["county"].fillna(df["home_address"].str.upper().str.findall(county_pattern).str[-1])
df["county"] = df["county"].str.title()

In [24]:
# Remove addresses that are just numbers
df.loc[df["company_address"].str.replace(" ", "").str.match(r"^\d+$", na=False), "company_address"] = None
df.loc[df["home_address"].str.replace(" ", "").str.match(r"^\d+$", na=False), "home_address"] = None


In [25]:
# 50s
def _remove_components_from_address(address, components):
    if pd.isna(address):
        return address
    for component in components:
        if pd.isna(component):
            continue
        component = str(component).replace("'", "")
        address = str(address).replace("'", "").replace(component, "")
    return address

df["company_address"] = df["company_address"].str.replace("united states", "", case=False)
df["company_address"] = df.apply(lambda x: _remove_components_from_address(x["company_address"], [x["company_zipcode"], x["company_city"], x["company_state"], x["company_zipfour"]]), axis=1)
df["company_address"] = df["company_address"].fillna("").str.split(",").apply(lambda x: " ".join(x).strip() if len(x) > 0 else None)

df["home_address"] = df["home_address"].str.replace("united states", "", case=False)
df["home_address"] = df.apply(lambda x: _remove_components_from_address(x["home_address"], [x["home_zipcode"], x["home_city"], x["home_state"], x["home_zipfour"]]), axis=1)
df["home_address"] = df["home_address"].fillna("").str.split(",").apply(lambda x: " ".join(x).strip() if len(x) > 0 else None)

In [26]:
# Helper functions for extracting valid social links - thsi can take up to 3 minutes to run
def extract_linkedin(val):
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if "linkedin.com/company" in s:
        return None  # Exclude company linkedin from personal
    if "linkedin" in s:
        return s
    return None

def extract_company_linkedin(val):
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if "linkedin.com/company" in s:
        return s
    return None

def extract_facebook(val):
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if "facebook" in s:
        return s
    return None

def extract_twitter(val):
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if "twitter" in s:
        return s
    return None

def is_valid_linkedin(val):
    return extract_linkedin(val) is not None

def is_valid_facebook(val):
    return extract_facebook(val) is not None

def is_valid_twitter(val):
    return extract_twitter(val) is not None

# For each row, find all valid values and move to correct columns.
# If the url is not linkedin/company_linkedin/facebook/twitter, put it under website
def reorganize_socials(row):
    sources = {
        "linkedin": row["linkedin"],
        "company_linkedin": row["company_linkedin"],
        "facebook": row["facebook"],
        "twitter": row["twitter"],
        "company_website": row["company_website"]
    }
    vals = list(sources.values())

    linkedin_vals = [extract_linkedin(v) for v in vals if extract_linkedin(v) is not None]
    company_linkedin_vals = [extract_company_linkedin(v) for v in vals if extract_company_linkedin(v) is not None]
    facebook_vals = [extract_facebook(v) for v in vals if extract_facebook(v) is not None]
    twitter_vals = [extract_twitter(v) for v in vals if extract_twitter(v) is not None]

    # Track which URLs are already assigned to socials
    assigned_urls = set()
    if linkedin_vals:
        assigned_urls.add(linkedin_vals[0])
    if company_linkedin_vals:
        assigned_urls.add(company_linkedin_vals[0])
    if facebook_vals:
        assigned_urls.add(facebook_vals[0])
    if twitter_vals:
        assigned_urls.add(twitter_vals[0])

    # Assign the first valid for each, or None
    row["linkedin"] = linkedin_vals[0] if linkedin_vals else None
    row["company_linkedin"] = company_linkedin_vals[0] if company_linkedin_vals else None
    row["facebook"] = facebook_vals[0] if facebook_vals else None
    row["twitter"] = twitter_vals[0] if twitter_vals else None

    # Now handle website: any http(s) url not used for above social fields
    def is_url(val):
        if pd.isna(val):
            return False
        s = str(val).strip().lower()
        return s.startswith("http") or "www" in s or ".co" in s
    website_candidates = []
    for v in vals:
        if is_url(v):
            s = str(v).strip().lower()
            # Not any valid social
            if (
                (extract_linkedin(s) is None)
                and (extract_company_linkedin(s) is None)
                and (extract_facebook(s) is None)
                and (extract_twitter(s) is None)
            ):
                website_candidates.append(s)
    # Set to website if one found
    row["company_website"] = website_candidates[0] if website_candidates else None

    return row

df = df.apply(reorganize_socials, axis=1)


In [27]:
# Fix misspellings in job titles, can take up to 30s to run
job_title_filters = {
    "Owner": [
        'owner',
        'onwer',
        'ownwer',
        'owneer',
        'ownder',
        'ownrer',
        'owmer',
        'ownew',
        'onwer',
        'owener',
        'ownr',
        'ower',
    ],
    "President": [
        'president',
        'presiden',
        'presdient',
        'presidnet',
        'presidant',
        'pressident',
        'presient',
        'presudent',
        'presidnt',
        'presedent',
        'pressdent',
        'preisdent',
        'pesident',
        'presid',
        'pres',
        'prez',
    ],
    "Chief Executive Officer": [
        'ceo',
        'chiefexecutiveofficer',
        'chiefexectutiveofficer',
        'chiefexecutingofficer',
        'chiefexectiveofficer',
        'chiefexcutiveofficer',
        'chiefexectuiveofficer',
        'chiefexecutionofficer',
        'chiefofexecutiveofficer',
        'chiefexecutiveoffice',
        'chiefexecutive',
        'cheifexecutive',
    ],
    "Founder": [
        'founder',
        'foundr',
        'foudner',
        'foudnerr',
        'foudner',
        'foudner',
        'foudner',
        'funder',
    ]
}

# Clean up job titles by replacing misspellings with canonical versions, can take up to 50s to run
for canonical, variations in job_title_filters.items():
    for variation in variations:
        # Use word boundaries to avoid partial matches, case-insensitive
        pattern = r'\b' + re.escape(variation) + r'\b'
        mask = df["job_title"].str.contains(pattern, case=False, regex=True, na=False)
        if mask.any():
            df.loc[mask, "job_title"] = df.loc[mask, "job_title"].str.replace(pattern, canonical, case=False, regex=True)
            break  # Move onto the next canonical once a variation is found



In [28]:
# Load company name mapping
with open("csv_outputs/company_name_clean/company_name_mapping.json", "r", encoding="utf-8") as f:
    company_name_mapping = json.load(f)

# Use map - much faster than replace for large dictionaries
df["company_name"] = df["company_name"].map(company_name_mapping).fillna(df["company_name"])

In [29]:
test_mobile_exp = df[(df["mobile_phone"].fillna("").str.contains('e+')) & (df["mobile_phone"].notna())]
if len(test_mobile_exp) > 0:
    print("TEST FAILED: MOBILE EXP")
else:
    print("TEST PASSED: MOBILE EXP")

test_landline_exp = df[(df["landline_phone"].fillna("").str.contains('e+')) & (df["landline_phone"].notna())]
if len(test_landline_exp) > 0:
    print("TEST FAILED: LANDLINE EXP")
else:
    print("TEST PASSED: LANDLINE EXP")

test_work_exp = df[(df["work_phone"].fillna("").str.contains('e+')) & (df["work_phone"].notna())]
if len(test_work_exp) > 0:
    print("TEST FAILED: WORK EXP")
else:
    print("TEST PASSED: WORK EXP")

test_email = df[(~df["email"].fillna("").str.contains("@")) & (df["email"].notna())]
if len(test_email) > 0:
    print("TEST FAILED: EMAIL")
else:
    print("TEST PASSED: EMAIL")

test_linkedin = df[(~df["linkedin"].fillna("").str.contains("linkedin")) & (df["linkedin"].notna())]
if len(test_linkedin) > 0:
    print("TEST FAILED: LINKEDIN")
else:
    print("TEST PASSED: LINKEDIN")
test_company_linkedin = df[(~df["company_linkedin"].fillna("").str.contains("linkedin")) & (df["company_linkedin"].notna())]
if len(test_company_linkedin) > 0:
    print("TEST FAILED: COMPANY LINKEDIN")
else:
    print("TEST PASSED: COMPANY LINKEDIN")

test_facebook = df[(~df["facebook"].fillna("").str.contains("facebook")) & (df["facebook"].notna())]
if len(test_facebook) > 0:
    print("TEST FAILED: FACEBOOK")
else:
    print("TEST PASSED: FACEBOOK")

test_twitter = df[(~df["twitter"].fillna("").str.contains("twitter")) & (df["twitter"].notna())]
if len(test_twitter) > 0:
    print("TEST FAILED: TWITTER")
else:
    print("TEST PASSED: TWITTER")

test_gender = df[(~df["gender"].isin(["M", "F"])) & (df["gender"].notna())]
if len(test_gender) > 0:
    print("TEST FAILED: GENDER")
else:
    print("TEST PASSED: GENDER")

test_email = df[(~df["email"].fillna("").str.contains("@")) & (df["email"].notna())]
if len(test_email) > 0:
    print(f"TEST FAILED: EMAIL {len(test_email)}")
else:
    print("TEST PASSED: EMAIL")




TEST PASSED: MOBILE EXP
TEST PASSED: LANDLINE EXP
TEST PASSED: WORK EXP
TEST PASSED: EMAIL
TEST PASSED: LINKEDIN
TEST PASSED: COMPANY LINKEDIN
TEST PASSED: FACEBOOK
TEST PASSED: TWITTER
TEST PASSED: GENDER
TEST PASSED: EMAIL


In [30]:
datetime_now = datetime.now().strftime("%Y-%m-%d")
df = df.sort_values(by=["company_name", "first_name", "last_name"])

df.to_csv(f"csv_outputs/clean_data_{datetime_now}.csv", index=False)

